### Colab Mount

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    from google.colab import drive

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    src_dir = Path("/content/drive/MyDrive/Luca/Research/AI-in-Science-Lab/greedy_stacked_autoencoders/src")
else:
    cwd = Path.cwd().resolve()
    src_dir = cwd if (cwd / "models").is_dir() else cwd / "greedy_stacked_autoencoders" / "src"

os.chdir(src_dir)
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

### Setup

In [ ]:
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from tqdm.auto import tqdm

from datasets import get_data_loader, get_patch_shape
from models import WTA_CONV_Greedy

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {DEVICE}")

In [ ]:
DEFAULT_CONFIG = {
    "project": "greedy-stacked-autoencoders",
    "dataset": "cifar10_patches_color",
    "architecture_type": "wta_conv_greedy",
    "learning_rate": 1e-3,
    "epochs": 3,
    "batch_size": 64,
    "hidden_channels": [64, 64],
    "k_spatial": 0.2,
    "k_population": None,
    "k_lifetime": 0.2,
    "lambda_strength": 1e-2,
    "kernel_size": 5,
    "correlation_mode": "mean",
    "comparison_mode": "both",
    "normalization_mode": "chnorm_relu",
    "postcomp_mode": "thresh",
    "correlation_loss": "max",
    "feature_map_mode": "post_wta",
    "log_every_steps": 100,
    "max_eval_batches": 25,
}

### Redundancy Loss

In [ ]:
class RedundancyReconstructionLoss(nn.Module):
    def __init__(
        self,
        lambda_strength: float,
        *,
        kernel_size: int = 3,
        correlation_mode: str = "mean",
        comparison_mode: str = "both",
        normalization_mode: str = "chnorm_relu",
        postcomp_mode: str = "thresh",
        correlation_loss: str = "max",
    ) -> None:
        super().__init__()

        kernel_size = int(kernel_size)
        if kernel_size <= 0 or (kernel_size % 2) == 0:
            raise ValueError(f"kernel_size must be an odd positive int, got {kernel_size}")

        self.lambda_strength = float(lambda_strength)
        self.kernel_radius = (kernel_size - 1) // 2
        self.correlation_mode = correlation_mode
        self.comparison_mode = comparison_mode
        self.normalization_mode = normalization_mode
        self.postcomp_mode = postcomp_mode
        self.correlation_loss = correlation_loss
        self.mse = nn.MSELoss()

        self.last_mse_loss = torch.tensor(0.0)
        self.last_corr_total = torch.tensor(0.0)
        self.last_corr_by_layer: dict[str, torch.Tensor] = {}

    def forward(
        self,
        reconstruction: torch.Tensor,
        target: torch.Tensor,
        feature_maps: list[tuple[str, torch.Tensor]],
    ) -> torch.Tensor:
        mse_loss = self.mse(reconstruction, target)
        corr_total = reconstruction.new_zeros(())
        corr_by_layer: dict[str, torch.Tensor] = {}

        for idx, feature_map_item in enumerate(feature_maps):
            layer_name, feature_map = self._parse_feature_map(feature_map_item, idx=idx)
            corr_mat = self.luca_fn(
                feature_map=feature_map,
                kernel_radius=self.kernel_radius,
                correlation_mode=self.correlation_mode,
                comparison_mode=self.comparison_mode,
                normalization_mode=self.normalization_mode,
                postcomp_mode=self.postcomp_mode,
            )
            corr_val = self._reduce_corr_matrix(corr_mat, self.correlation_loss)
            corr_total = corr_total + corr_val
            corr_by_layer[layer_name] = corr_val.detach()

        self.last_mse_loss = mse_loss.detach()
        self.last_corr_total = corr_total.detach()
        self.last_corr_by_layer = corr_by_layer

        return mse_loss + (self.lambda_strength * corr_total)

    @staticmethod
    def _parse_feature_map(
        feature_map_item: tuple[str, torch.Tensor] | torch.Tensor,
        *,
        idx: int,
    ) -> tuple[str, torch.Tensor]:
        if isinstance(feature_map_item, torch.Tensor):
            return f"fm_{idx}", feature_map_item
        if isinstance(feature_map_item, (tuple, list)) and len(feature_map_item) >= 2:
            return str(feature_map_item[0]), feature_map_item[1]
        raise TypeError("feature_maps must contain tensors or (name, tensor) pairs.")

    @staticmethod
    def _reduce_corr_matrix(corr_mat: torch.Tensor, correlation_loss: str) -> torch.Tensor:
        match correlation_loss:
            case "sum":
                return corr_mat.sum()
            case "max":
                return corr_mat.amax()
            case _:
                raise ValueError("correlation_loss must be 'sum' or 'max'.")

    @staticmethod
    def luca_fn(
        feature_map: torch.Tensor,
        kernel_radius: int,
        correlation_mode: str,
        comparison_mode: str,
        normalization_mode: str,
        postcomp_mode: str,
    ) -> torch.Tensor:
        if correlation_mode not in {"mean", "max"}:
            raise ValueError("correlation_mode must be 'mean' or 'max'.")
        if comparison_mode not in {"shift", "batch", "both"}:
            raise ValueError("comparison_mode must be 'shift', 'batch', or 'both'.")
        if normalization_mode not in {"relu", "relu_log", "chnorm_relu"}:
            raise ValueError("normalization_mode must be 'relu', 'relu_log', or 'chnorm_relu'.")
        if postcomp_mode not in {"squared", "thresh"}:
            raise ValueError("postcomp_mode must be 'squared' or 'thresh'.")

        eps = 1e-6
        batch_size, channels, _, _ = feature_map.shape
        fmap = feature_map
        radius = kernel_radius

        if normalization_mode == "relu":
            fmap = F.relu(fmap)
        elif normalization_mode == "relu_log":
            fmap = torch.log1p(F.relu(fmap))
        elif normalization_mode == "chnorm_relu":
            ch_mean = fmap.mean(dim=(0, 2, 3), keepdim=True)
            ch_std = fmap.std(dim=(0, 2, 3), keepdim=True, unbiased=False)
            fmap = F.relu((fmap - ch_mean) / (ch_std + eps))

        patch_hw = (2 * radius) + 1
        patch_len = patch_hw * patch_hw
        patches = F.unfold(fmap, kernel_size=patch_hw, padding=radius)
        patches = patches.view(batch_size, channels, patch_len, -1)
        center = patches[:, :, patch_len // 2, :]

        if comparison_mode == "shift":
            corr = torch.einsum("bip,bjkp->ijbp", center, patches) / patch_len
        elif comparison_mode == "batch":
            corr = torch.einsum("bip,bjkp->ijkp", center, patches) / batch_size
        else:
            corr = torch.einsum("bip,bjkp->ijp", center, patches) / (patch_len * batch_size)

        if postcomp_mode == "squared":
            corr = corr * corr
        elif postcomp_mode == "thresh":
            corr = F.relu(corr)

        if correlation_mode == "max":
            corr = corr.amax(dim=tuple(range(2, corr.ndim)))
        elif correlation_mode == "mean":
            corr = corr.mean(dim=tuple(range(2, corr.ndim)))

        mask = torch.triu(
            torch.ones((channels, channels), dtype=torch.bool, device=feature_map.device),
            diagonal=1,
        )
        return corr * mask

### Training

In [ ]:
def get_config_name(cfg: dict[str, Any]) -> str:
    return (
        f"{cfg['architecture_type']}"
        f"-ks_{cfg['kernel_size']}"
        f"-lam_{cfg['lambda_strength']}"
        f"-cm_{cfg['correlation_mode']}"
        f"-cmp_{cfg['comparison_mode']}"
        f"-nm_{cfg['normalization_mode']}"
        f"-pc_{cfg['postcomp_mode']}"
        f"-cl_{cfg['correlation_loss']}"
        f"-wta_{cfg['feature_map_mode']}"
        f"-lr_{cfg['learning_rate']}"
    )


def _capitalize_top_folder(key: str) -> str:
    if "/" not in key:
        return key
    top, rest = key.split("/", 1)
    if not top:
        return key
    return f"{top[:1].upper()}{top[1:]}" + f"/{rest}"


def wandb_log(payload: dict[str, object], *, step: int | None = None) -> None:
    payload = {_capitalize_top_folder(key): value for key, value in payload.items()}
    if step is None:
        wandb.log(payload)
        return
    wandb.log(payload, step=step)


def build_model(cfg: dict[str, Any]) -> WTA_CONV_Greedy:
    channels, height, width = get_patch_shape(cfg["dataset"])
    return WTA_CONV_Greedy(
        dim=(channels, height, width),
        hidden_channels=tuple(cfg["hidden_channels"]),
        k_spatial=float(cfg["k_spatial"]),
        k_population=cfg.get("k_population"),
        k_lifetime=cfg.get("k_lifetime"),
        total_epochs=int(cfg["epochs"]),
        dataset_size=int(cfg["dataset_size"]),
        a=5.0,
        feature_map_mode=cfg["feature_map_mode"],
    ).to(DEVICE)


def log_feature_map_stats(feature_maps: list[tuple[str, torch.Tensor]], *, step: int) -> None:
    payload: dict[str, float] = {}
    for layer_name, feature_map in feature_maps:
        fmap = feature_map.detach()
        payload[f"Feature_map/{layer_name}/mean"] = float(fmap.mean().cpu())
        payload[f"Feature_map/{layer_name}/max"] = float(fmap.amax().cpu())
        payload[f"Feature_map/{layer_name}/active_fraction"] = float((fmap > 1e-5).to(fmap.dtype).mean().cpu())
    if payload:
        wandb_log(payload, step=step)

In [ ]:
def train(config: dict[str, Any]) -> dict[str, float]:
    run = wandb.init(project=config["project"], config=config)
    cfg = dict(wandb.config)
    run.name = get_config_name(cfg)

    train_loader = get_data_loader(cfg["dataset"], train=True, batch_size=int(cfg["batch_size"]))
    test_loader = get_data_loader(cfg["dataset"], train=False, batch_size=int(cfg["batch_size"]))
    cfg["dataset_size"] = len(train_loader.dataset)

    model = build_model(cfg)
    criterion = RedundancyReconstructionLoss(
        lambda_strength=cfg["lambda_strength"],
        kernel_size=int(cfg["kernel_size"]),
        correlation_mode=cfg["correlation_mode"],
        comparison_mode=cfg["comparison_mode"],
        normalization_mode=cfg["normalization_mode"],
        postcomp_mode=cfg["postcomp_mode"],
        correlation_loss=cfg["correlation_loss"],
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(cfg["learning_rate"]))

    def eval_model(epoch: int) -> tuple[float, float]:
        model.eval()
        loss_sum = 0.0
        mse_sum = 0.0
        n = 0
        with torch.no_grad():
            for batch_idx, (xb, _) in enumerate(test_loader):
                if batch_idx >= int(cfg["max_eval_batches"]):
                    break
                xb = xb.to(DEVICE, non_blocking=True)
                reconstruction, feature_maps = model(xb, epoch=epoch, inputs_processed_in_epoch=0)
                loss = criterion(reconstruction, xb, feature_maps)
                batch_size = int(xb.shape[0])
                loss_sum += float(loss.detach().cpu()) * batch_size
                mse_sum += float(criterion.last_mse_loss.detach().cpu()) * batch_size
                n += batch_size
        model.train()
        return loss_sum / max(1, n), mse_sum / max(1, n)

    global_step = 0
    log_every_steps = int(cfg["log_every_steps"])
    eval_interval_steps = max(1, len(train_loader) // 4)

    for epoch in tqdm(range(int(cfg["epochs"])), desc="Epochs"):
        model.train()
        inputs_processed_in_epoch = 0
        epoch_loss = 0.0

        for step_in_epoch, (xb, _) in enumerate(train_loader):
            xb = xb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            reconstruction, feature_maps = model(
                xb,
                epoch=epoch,
                inputs_processed_in_epoch=inputs_processed_in_epoch,
            )
            loss = criterion(reconstruction, xb, feature_maps)
            loss.backward()
            optimizer.step()

            batch_size = int(xb.shape[0])
            inputs_processed_in_epoch += batch_size
            epoch_loss += float(loss.detach().cpu())

            if global_step % log_every_steps == 0:
                log_payload: dict[str, float | int] = {
                    "Train/loss": float(loss.detach().cpu()),
                    "Train/mse": float(criterion.last_mse_loss.detach().cpu()),
                    "Losses/correlation_total": float(criterion.last_corr_total.detach().cpu()),
                    "General/epoch": epoch,
                    "General/step": global_step,
                }
                if hasattr(model, "last_k"):
                    log_payload["General/last_k"] = int(model.last_k)
                for layer_name, layer_loss in criterion.last_corr_by_layer.items():
                    log_payload[f"Losses/{layer_name}"] = float(layer_loss.detach().cpu())
                wandb_log(log_payload, step=global_step)
                log_feature_map_stats(feature_maps, step=global_step)

            global_step += 1

            if (step_in_epoch + 1) % eval_interval_steps == 0:
                test_loss, test_mse = eval_model(epoch)
                wandb_log(
                    {
                        "Test/loss": test_loss,
                        "Test/mse": test_mse,
                        "General/epoch": epoch,
                        "General/step": global_step,
                    },
                    step=global_step,
                )

        train_loss = epoch_loss / max(1, len(train_loader))
        test_loss, test_mse = eval_model(epoch)
        wandb_log(
            {
                "Train/epoch_loss": train_loss,
                "Test/loss": test_loss,
                "Test/mse": test_mse,
                "General/epoch": epoch,
                "General/step": global_step,
            },
            step=global_step,
        )

    final_loss, final_mse = eval_model(int(cfg["epochs"]) - 1)
    metrics = {"Test/loss": final_loss, "Test/mse": final_mse}
    wandb.finish()
    return metrics

### Sweeps

In [ ]:
def sweep_train() -> None:
    train(DEFAULT_CONFIG)


SWEEP_PROJECT = DEFAULT_CONFIG["project"]

SWEEP_CONFIG_MAIN = {
    "method": "grid",
    "metric": {"name": "Test/mse", "goal": "minimize"},
    "parameters": {
        "lambda_strength": {
            "values": [0.0, 1e-2]
        },
        "learning_rate": {
            "value": 1e-3
        },
        "epochs": {
            "value": 3
        },
        "batch_size": {
            "value": 64
        },
        "hidden_channels": {
            "value": [64, 64]
        },
        "k_spatial": {
            "value": 0.2
        },
        "k_population": {
            "value": None
        },
        "k_lifetime": {
            "value": 0.2
        },
        "kernel_size": {
            "value": 5
        },
        "correlation_mode": {
            "value": "mean"
        },
        "comparison_mode": {
            "value": "both"
        },
        "normalization_mode": {
            "value": "chnorm_relu"
        },
        "postcomp_mode": {
            "value": "thresh"
        },
        "correlation_loss": {
            "value": "max"
        },
        "feature_map_mode": {
            "value": "post_wta"
        },
        "dataset": {
            "value": "cifar10_patches_color"
        },
        "architecture_type": {
            "value": "wta_conv_greedy"
        },
        "project": {
            "value": SWEEP_PROJECT
        },
        "log_every_steps": {
            "value": 100
        },
        "max_eval_batches": {
            "value": 25
        },
    },
}


def run_sweep(*, sweep_id: str | None = None, sweep_config: dict[str, Any] = SWEEP_CONFIG_MAIN) -> str:
    if sweep_id is None:
        sweep_id = wandb.sweep(sweep_config, project=SWEEP_PROJECT)
    wandb.agent(sweep_id, function=sweep_train)
    return sweep_id

### Run

In [ ]:
metrics = train(DEFAULT_CONFIG)
metrics